# Speaker Diarization + Per-Speaker Transcription

Split audio by speaker using **pyannote 3.1** for diarization and **faster-whisper large-v3** for transcription.

**Pipeline:**
1. Extract audio from video (ffmpeg)
2. Speaker diarization (pyannote) — who speaks when
3. Inspect diarization results
4. Transcription (faster-whisper) — what is said
5. Merge diarization + transcript — who said what
6. Split audio by speaker (ffmpeg)
7. Save all outputs (JSON, TXT, SRT, per-speaker WAV+TXT)

## 1. Setup & Imports

In [1]:
# Install dependencies (run once)
# !pip install faster-whisper pyannote.audio

In [2]:
import gc
import json
import os
import subprocess
import time
from pathlib import Path

import torch

# Fix: use system ffmpeg if conda ffmpeg has broken libopenh264
try:
    subprocess.run(["ffmpeg", "-version"], capture_output=True, check=True)
except (subprocess.CalledProcessError, OSError):
    os.environ["PATH"] = "/usr/bin:" + os.environ.get("PATH", "")
    print("Using system ffmpeg (conda ffmpeg has broken libopenh264)")

# HuggingFace token for pyannote (gated model)
os.environ["HF_TOKEN"] = "YOUR_HF_TOKEN"
HF_TOKEN = os.environ["HF_TOKEN"]

print(f"Torch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
print(f"HF_TOKEN: ...{HF_TOKEN[-4:]}")

Using system ffmpeg (conda ffmpeg has broken libopenh264)
Torch version: 2.9.1+cu128
CUDA available: False
Device: cpu
HF_TOKEN: ...bPNK


In [3]:
# Paths
PROJECT_ROOT = Path(".").resolve().parent
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

# Input: video file
VIDEO_FILE = DATA_DIR / "le20250615_real_111424_113400055_20250615_200420_35.mp4"
WAV_FILE = VIDEO_FILE.with_suffix(".wav")

print(f"Video: {VIDEO_FILE.name}")
print(f"Size:  {VIDEO_FILE.stat().st_size / 1024 / 1024:.1f} MB")
print(f"WAV:   {WAV_FILE.name}")

Video: le20250615_real_111424_113400055_20250615_200420_35.mp4
Size:  631.9 MB
WAV:   le20250615_real_111424_113400055_20250615_200420_35.wav


## 2. Extract Audio from Video

Extract audio as WAV 16kHz mono (optimal for both pyannote and Whisper).

In [4]:
# Extract first 3 minutes only (for fast testing on CPU)
EXTRACT_DURATION = 180  # seconds (3 minutes)

if WAV_FILE.exists():
    # Check if existing WAV is already the right length
    probe = subprocess.run(
        ["ffprobe", "-v", "quiet", "-show_entries", "format=duration",
         "-of", "default=noprint_wrappers=1:nokey=1", str(WAV_FILE)],
        capture_output=True, text=True,
    )
    existing_duration = float(probe.stdout.strip())
    if abs(existing_duration - EXTRACT_DURATION) < 5:
        print(f"WAV already exists: {WAV_FILE.name} ({existing_duration:.0f}s, {WAV_FILE.stat().st_size / 1024 / 1024:.1f} MB)")
    else:
        print(f"Existing WAV is {existing_duration:.0f}s, re-extracting {EXTRACT_DURATION}s...")
        WAV_FILE.unlink()

if not WAV_FILE.exists():
    print(f"Extracting first {EXTRACT_DURATION}s: {VIDEO_FILE.name} -> {WAV_FILE.name}")
    t0 = time.time()
    result = subprocess.run(
        [
            "ffmpeg", "-i", str(VIDEO_FILE),
            "-t", str(EXTRACT_DURATION),
            "-vn", "-acodec", "pcm_s16le", "-ar", "16000", "-ac", "1",
            "-y", str(WAV_FILE),
        ],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        print(f"ERROR: {result.stderr[-500:]}")
    else:
        elapsed = time.time() - t0
        print(f"Done in {elapsed:.1f}s")
        print(f"WAV size: {WAV_FILE.stat().st_size / 1024 / 1024:.1f} MB")

# Get audio duration
probe = subprocess.run(
    ["ffprobe", "-v", "quiet", "-show_entries", "format=duration",
     "-of", "default=noprint_wrappers=1:nokey=1", str(WAV_FILE)],
    capture_output=True, text=True,
)
AUDIO_DURATION = float(probe.stdout.strip())
minutes = int(AUDIO_DURATION // 60)
seconds = AUDIO_DURATION % 60
print(f"Duration: {minutes}m {seconds:.1f}s ({AUDIO_DURATION:.1f}s total)")

Existing WAV is 3433s, re-extracting 180s...
Extracting first 180s: le20250615_real_111424_113400055_20250615_200420_35.mp4 -> le20250615_real_111424_113400055_20250615_200420_35.wav


Done in 0.5s
WAV size: 5.5 MB
Duration: 3m 0.0s (180.0s total)


## 3. Speaker Diarization (pyannote 3.1)

Identify **who speaks when**. pyannote is language-agnostic (works on acoustic features, not text).

Prerequisites:
- Accept model license at https://huggingface.co/pyannote/speaker-diarization-3.1
- Accept model license at https://huggingface.co/pyannote/segmentation-3.0

In [5]:
from pyannote.audio import Pipeline

print("Loading pyannote speaker-diarization-3.1 pipeline...")
t0 = time.time()

diarization_pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-3.1",
    token=HF_TOKEN,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
diarization_pipeline.to(device)

elapsed = time.time() - t0
print(f"Pipeline loaded on {device} in {elapsed:.1f}s")

/home/airsrg/mambaforge/lib/python3.10/site-packages/pyannote/audio/core/io.py:47: UserWarning: 
torchcodec is not installed correctly so built-in audio decoding will fail. Solutions are:
* use audio preloaded in-memory as a {'waveform': (channel, time) torch.Tensor, 'sample_rate': int} dictionary;
* fix torchcodec installation. Error message was:

Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6, 7, and 8, and we attempt to load libtorchcodec
             for each of those versions. Errors for versions not installed on
             your system are expected; only the error for your installed FFmpeg
             version is relevant. On Windows, ensure you've installed the
             "full-shared" version which ships DLLs.
          2. The PyTorch version (2.9.1+cu128) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
             tabl

Loading pyannote speaker-diarization-3.1 pipeline...


Pipeline loaded on cpu in 4.2s


/home/airsrg/mambaforge/lib/python3.10/site-packages/torch/cuda/__init__.py:827: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [6]:
# Pre-load audio with scipy (workaround: torchaudio/torchcodec need CUDA libs not present)
from scipy.io import wavfile

print(f"Loading audio: {WAV_FILE.name}")
sample_rate, audio_np = wavfile.read(str(WAV_FILE))
# Convert int16 WAV to float32 tensor, shape (1, num_samples) for pyannote
waveform = torch.from_numpy(audio_np.astype("float32") / 32768.0).unsqueeze(0)
print(f"  Waveform shape: {waveform.shape}, sample_rate: {sample_rate}")

# Prepare audio dict for pyannote (bypasses torchcodec)
audio_input = {"waveform": waveform, "sample_rate": sample_rate}

# Run diarization
# Set num_speakers if you know how many speakers are in the audio (optional)
NUM_SPEAKERS = None  # e.g., 2, 3 — or None for auto-detection

print(f"\nRunning diarization...")
print(f"Audio duration: {AUDIO_DURATION:.0f}s — this may take a while on CPU...")
t0 = time.time()

diarization_kwargs = {}
if NUM_SPEAKERS is not None:
    diarization_kwargs["num_speakers"] = NUM_SPEAKERS
    print(f"Forcing num_speakers={NUM_SPEAKERS}")

diarization_result = diarization_pipeline(audio_input, **diarization_kwargs)

elapsed = time.time() - t0
print(f"Diarization complete in {elapsed:.1f}s ({elapsed/AUDIO_DURATION:.1f}x real-time)")

# Extract segments (pyannote 4.x: result.speaker_diarization is the Annotation)
annotation = diarization_result.speaker_diarization

diar_segments = []
for turn, _, speaker in annotation.itertracks(yield_label=True):
    diar_segments.append({
        "speaker": speaker,
        "start": round(turn.start, 3),
        "end": round(turn.end, 3),
    })

speakers = sorted(set(s["speaker"] for s in diar_segments))
print(f"\nFound {len(speakers)} speakers: {', '.join(speakers)}")
print(f"Total segments: {len(diar_segments)}")

# Free the waveform
del waveform, audio_input, audio_np

Loading audio: le20250615_real_111424_113400055_20250615_200420_35.wav
  Waveform shape: torch.Size([1, 2880001]), sample_rate: 16000

Running diarization...
Audio duration: 180s — this may take a while on CPU...


/home/airsrg/mambaforge/lib/python3.10/site-packages/pyannote/audio/models/blocks/pooling.py:103: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1857.)
  std = sequences.std(dim=-1, correction=1)


Diarization complete in 414.4s (2.3x real-time)

Found 4 speakers: SPEAKER_00, SPEAKER_01, SPEAKER_02, SPEAKER_03
Total segments: 19


In [7]:
# Release pyannote to free memory before loading Whisper
del diarization_pipeline
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("pyannote pipeline released from memory")

pyannote pipeline released from memory


/home/airsrg/mambaforge/lib/python3.10/site-packages/torch/cuda/__init__.py:827: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


## 4. Inspect Diarization Results

Review speaker statistics and timeline before proceeding. If the speaker count looks wrong, go back to Cell 3 and set `NUM_SPEAKERS` explicitly.

In [8]:
def fmt_time(seconds):
    """Format seconds as MM:SS.ss"""
    m, s = divmod(seconds, 60)
    return f"{int(m):02d}:{s:05.2f}"

def fmt_srt(seconds):
    """Format seconds as HH:MM:SS,mmm for SRT"""
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    ms = int((seconds % 1) * 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

# Per-speaker statistics
print(f"{'Speaker':<15} {'Segments':>8} {'Total Time':>12} {'Avg Segment':>12}")
print("-" * 50)
for speaker in speakers:
    segs = [s for s in diar_segments if s["speaker"] == speaker]
    total_time = sum(s["end"] - s["start"] for s in segs)
    avg_len = total_time / len(segs) if segs else 0
    print(f"{speaker:<15} {len(segs):>8} {fmt_time(total_time):>12} {avg_len:>11.1f}s")

print(f"\n{'Total':<15} {len(diar_segments):>8} {fmt_time(AUDIO_DURATION):>12}")

# Preview: first 10 segments
print(f"\n--- First 10 diarization segments ---")
for seg in diar_segments[:10]:
    duration = seg["end"] - seg["start"]
    print(f"  [{fmt_time(seg['start'])} - {fmt_time(seg['end'])}] ({duration:.1f}s) {seg['speaker']}")

if len(diar_segments) > 10:
    print(f"  ... ({len(diar_segments) - 10} more segments)")

# Preview: last 5 segments
print(f"\n--- Last 5 diarization segments ---")
for seg in diar_segments[-5:]:
    duration = seg["end"] - seg["start"]
    print(f"  [{fmt_time(seg['start'])} - {fmt_time(seg['end'])}] ({duration:.1f}s) {seg['speaker']}")

Speaker         Segments   Total Time  Avg Segment
--------------------------------------------------
SPEAKER_00             2     00:05.91         3.0s
SPEAKER_01             2     00:01.11         0.6s
SPEAKER_02             2     00:01.45         0.7s
SPEAKER_03            13     00:06.97         0.5s

Total                 19     03:00.00

--- First 10 diarization segments ---
  [00:14.73 - 00:17.24] (2.5s) SPEAKER_00
  [00:17.45 - 00:20.84] (3.4s) SPEAKER_00
  [00:30.96 - 00:31.03] (0.1s) SPEAKER_03
  [00:31.28 - 00:31.93] (0.6s) SPEAKER_03
  [00:31.98 - 00:32.14] (0.2s) SPEAKER_03
  [00:33.55 - 00:33.71] (0.2s) SPEAKER_03
  [00:33.97 - 00:34.03] (0.1s) SPEAKER_03
  [00:34.37 - 00:36.30] (1.9s) SPEAKER_03
  [00:36.33 - 00:36.38] (0.1s) SPEAKER_03
  [00:39.60 - 00:40.73] (1.1s) SPEAKER_02
  ... (9 more segments)

--- Last 5 diarization segments ---
  [00:57.54 - 00:57.71] (0.2s) SPEAKER_03
  [01:14.42 - 01:15.48] (1.1s) SPEAKER_03
  [01:18.28 - 01:19.31] (1.0s) SPEAKER_01
  [01:19.

## 5. Transcription (faster-whisper large-v3)

Transcribe the **full audio** with faster-whisper (4-6x faster than openai-whisper via CTranslate2).
Using `int8` compute type for CPU efficiency.

In [9]:
from faster_whisper import WhisperModel

WHISPER_MODEL = "large-v3"  # best for Bulgarian; alternatives: "medium", "small"
COMPUTE_TYPE = "int8"       # CPU-optimized quantization

print(f"Loading faster-whisper model: {WHISPER_MODEL} (compute_type={COMPUTE_TYPE})")
t0 = time.time()

whisper_model = WhisperModel(
    WHISPER_MODEL,
    device="cpu",
    compute_type=COMPUTE_TYPE,
)

elapsed = time.time() - t0
print(f"Model loaded in {elapsed:.1f}s")

Loading faster-whisper model: large-v3 (compute_type=int8)


model.bin:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

vocabulary.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

Model loaded in 655.5s


In [10]:
# Transcribe full audio
print(f"Transcribing: {WAV_FILE.name} ({AUDIO_DURATION:.0f}s)")
print("This may take a while on CPU...")
t0 = time.time()

segments_iter, info = whisper_model.transcribe(
    str(WAV_FILE),
    language="bg",
    task="transcribe",
    beam_size=5,
    word_timestamps=True,
    condition_on_previous_text=False,  # prevent hallucination cascades
    no_speech_threshold=0.6,           # stricter silence detection
    compression_ratio_threshold=2.4,   # reject garbled segments
)

# Collect segments from iterator
whisper_segments = []
for seg in segments_iter:
    whisper_segments.append({
        "start": seg.start,
        "end": seg.end,
        "text": seg.text.strip(),
    })

elapsed = time.time() - t0
print(f"Transcription complete in {elapsed:.1f}s ({elapsed/AUDIO_DURATION:.1f}x real-time)")
print(f"Segments: {len(whisper_segments)}")
print(f"Detected language: {info.language} (probability: {info.language_probability:.2f})")

# Preview first 10 segments
print(f"\n--- First 10 transcript segments ---")
for seg in whisper_segments[:10]:
    print(f"  [{fmt_time(seg['start'])} - {fmt_time(seg['end'])}] {seg['text']}")
if len(whisper_segments) > 10:
    print(f"  ... ({len(whisper_segments) - 10} more segments)")

Transcribing: le20250615_real_111424_113400055_20250615_200420_35.wav (180s)
This may take a while on CPU...


Transcription complete in 324.2s (1.8x real-time)
Segments: 14
Detected language: bg (probability: 1.00)

--- First 10 transcript segments ---
  [00:15.00 - 00:17.10] Остройството не е позиционирано правилно.
  [00:17.40 - 00:19.64] Завъртете го така, че физическите бутони
  [00:19.64 - 00:20.50] да бъдат отгоре.
  [00:34.54 - 00:36.22] Трябва ли да се натиска нещо?
  [00:38.14 - 00:40.64] Готовите ви съще да се затворите и...
  [00:40.64 - 00:43.20] Тук е клипот, който се общи някакъв транс.
  [00:47.90 - 00:49.10] Обърнете телефона.
  [00:50.90 - 00:51.50] Така?
  [00:51.50 - 00:52.28] Не, не, не.
  [01:20.86 - 01:22.26] Време е екранта е тъмбар.
  ... (4 more segments)


In [11]:
# Release whisper model to free memory
del whisper_model
gc.collect()
print("Whisper model released from memory")

Whisper model released from memory


## 6. Merge Diarization + Transcript

Assign speaker labels to each transcript segment by finding which diarization segment overlaps with the transcript segment's midpoint.

In [12]:
# Merge: assign speaker labels to transcript segments
merged_segments = []

for wseg in whisper_segments:
    w_mid = (wseg["start"] + wseg["end"]) / 2
    best_speaker = None
    best_dist = float("inf")

    for dseg in diar_segments:
        if dseg["start"] <= w_mid <= dseg["end"]:
            best_speaker = dseg["speaker"]
            best_dist = 0
            break
        # Track nearest segment in case midpoint falls in a gap
        dist = min(abs(w_mid - dseg["start"]), abs(w_mid - dseg["end"]))
        if dist < best_dist:
            best_dist = dist
            best_speaker = dseg["speaker"]

    merged_segments.append({
        "speaker": best_speaker or "UNKNOWN",
        "start": wseg["start"],
        "end": wseg["end"],
        "text": wseg["text"],
    })

print(f"Merged {len(merged_segments)} segments with speaker labels")

# Preview: first 15 merged segments
print(f"\n--- Diarized transcript (first 15 segments) ---")
for seg in merged_segments[:15]:
    print(f"  [{fmt_time(seg['start'])} - {fmt_time(seg['end'])}] {seg['speaker']}: {seg['text']}")
if len(merged_segments) > 15:
    print(f"  ... ({len(merged_segments) - 15} more segments)")

# Per-speaker text preview
print(f"\n--- Per-speaker preview ---")
for speaker in speakers:
    segs = [s for s in merged_segments if s["speaker"] == speaker]
    total_text = " ".join(s["text"] for s in segs)
    print(f"\n{speaker} ({len(segs)} segments):")
    print(f"  {total_text[:200]}{'...' if len(total_text) > 200 else ''}")

Merged 14 segments with speaker labels

--- Diarized transcript (first 15 segments) ---
  [00:15.00 - 00:17.10] SPEAKER_00: Остройството не е позиционирано правилно.
  [00:17.40 - 00:19.64] SPEAKER_00: Завъртете го така, че физическите бутони
  [00:19.64 - 00:20.50] SPEAKER_00: да бъдат отгоре.
  [00:34.54 - 00:36.22] SPEAKER_03: Трябва ли да се натиска нещо?
  [00:38.14 - 00:40.64] SPEAKER_02: Готовите ви съще да се затворите и...
  [00:40.64 - 00:43.20] SPEAKER_03: Тук е клипот, който се общи някакъв транс.
  [00:47.90 - 00:49.10] SPEAKER_02: Обърнете телефона.
  [00:50.90 - 00:51.50] SPEAKER_03: Така?
  [00:51.50 - 00:52.28] SPEAKER_03: Не, не, не.
  [01:20.86 - 01:22.26] SPEAKER_01: Време е екранта е тъмбар.
  [01:26.90 - 01:27.74] SPEAKER_03: Вече си не е жив.
  [01:30.34 - 01:31.62] SPEAKER_03: Да, да, готово!
  [01:31.62 - 01:33.58] SPEAKER_03: Благодаря.
  [02:58.58 - 02:59.98] SPEAKER_03: Абонирайте се!

--- Per-speaker preview ---

SPEAKER_00 (3 segments):
  Остройството не е

## 7. Split Audio by Speaker

Extract per-speaker audio files using ffmpeg. Each speaker gets one WAV file containing all their segments concatenated.

In [13]:
# Create output directory
audio_stem = VIDEO_FILE.stem
speaker_output_dir = OUTPUT_DIR / f"{audio_stem}_speakers"
speaker_output_dir.mkdir(parents=True, exist_ok=True)

print(f"Output directory: {speaker_output_dir}")

# Split audio by speaker using ffmpeg atrim + concat filter
speaker_files = {}

for speaker in speakers:
    segs = [s for s in diar_segments if s["speaker"] == speaker]
    if not segs:
        continue

    speaker_suffix = speaker.replace("SPEAKER_", "speaker_")
    out_path = speaker_output_dir / f"{audio_stem}_{speaker_suffix}.wav"

    # Build ffmpeg filter_complex
    filter_parts = []
    concat_inputs = []
    for i, seg in enumerate(segs):
        label = f"s{i}"
        filter_parts.append(
            f"[0]atrim=start={seg['start']}:end={seg['end']},asetpts=PTS-STARTPTS[{label}]"
        )
        concat_inputs.append(f"[{label}]")

    filter_complex = "; ".join(filter_parts)
    filter_complex += f"; {''.join(concat_inputs)}concat=n={len(segs)}:v=0:a=1[out]"

    cmd = [
        "ffmpeg", "-i", str(WAV_FILE),
        "-filter_complex", filter_complex,
        "-map", "[out]",
        "-ar", "16000", "-ac", "1",
        "-y", str(out_path),
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"  ERROR splitting {speaker}: {result.stderr[-300:]}")
        continue

    duration = sum(s["end"] - s["start"] for s in segs)
    size_mb = out_path.stat().st_size / 1024 / 1024
    print(f"  {out_path.name}: {len(segs)} segments, {duration:.1f}s, {size_mb:.1f} MB")
    speaker_files[speaker] = out_path

print(f"\nSplit into {len(speaker_files)} speaker files")

Output directory: /home/airsrg/work/video_streaming/output/le20250615_real_111424_113400055_20250615_200420_35_speakers
  le20250615_real_111424_113400055_20250615_200420_35_speaker_00.wav: 2 segments, 5.9s, 0.2 MB
  le20250615_real_111424_113400055_20250615_200420_35_speaker_01.wav: 2 segments, 1.1s, 0.0 MB
  le20250615_real_111424_113400055_20250615_200420_35_speaker_02.wav: 2 segments, 1.5s, 0.0 MB


  le20250615_real_111424_113400055_20250615_200420_35_speaker_03.wav: 13 segments, 7.0s, 0.2 MB

Split into 4 speaker files


## 8. Save All Outputs

Save combined diarized transcript (JSON, TXT, SRT) and per-speaker transcripts (TXT).

In [14]:
# --- Combined diarized transcript ---

# JSON
json_path = speaker_output_dir / f"{audio_stem}_diarized.json"
json_data = {
    "audio_file": VIDEO_FILE.name,
    "whisper_model": WHISPER_MODEL,
    "num_speakers": len(speakers),
    "speakers": speakers,
    "audio_duration": AUDIO_DURATION,
    "segments": merged_segments,
}
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(json_data, f, ensure_ascii=False, indent=2)
print(f"Saved: {json_path.name}")

# TXT (human-readable with speaker labels)
txt_path = speaker_output_dir / f"{audio_stem}_diarized.txt"
with open(txt_path, "w", encoding="utf-8") as f:
    for seg in merged_segments:
        f.write(f"[{fmt_time(seg['start'])} - {fmt_time(seg['end'])}] {seg['speaker']}: {seg['text']}\n")
print(f"Saved: {txt_path.name}")

# SRT (subtitles with speaker prefix)
srt_path = speaker_output_dir / f"{audio_stem}_diarized.srt"
with open(srt_path, "w", encoding="utf-8") as f:
    for i, seg in enumerate(merged_segments, 1):
        f.write(f"{i}\n")
        f.write(f"{fmt_srt(seg['start'])} --> {fmt_srt(seg['end'])}\n")
        f.write(f"{seg['speaker']}: {seg['text']}\n\n")
print(f"Saved: {srt_path.name}")

# --- Per-speaker transcripts ---
for speaker in speakers:
    segs = [s for s in merged_segments if s["speaker"] == speaker]
    speaker_suffix = speaker.replace("SPEAKER_", "speaker_")
    per_txt = speaker_output_dir / f"{audio_stem}_{speaker_suffix}.txt"
    with open(per_txt, "w", encoding="utf-8") as f:
        for seg in segs:
            f.write(f"[{fmt_time(seg['start'])} - {fmt_time(seg['end'])}] {seg['text']}\n")
    print(f"Saved: {per_txt.name} ({len(segs)} segments)")

# --- Summary ---
print(f"\n{'='*60}")
print(f"DONE! {len(speakers)} speakers identified.")
print(f"All outputs in: {speaker_output_dir}/")
print(f"{'='*60}")
print()
for f in sorted(speaker_output_dir.glob("*")):
    print(f"  {f.name:<60s} {f.stat().st_size / 1024:>7.1f} KB")

Saved: le20250615_real_111424_113400055_20250615_200420_35_diarized.json
Saved: le20250615_real_111424_113400055_20250615_200420_35_diarized.txt
Saved: le20250615_real_111424_113400055_20250615_200420_35_diarized.srt
Saved: le20250615_real_111424_113400055_20250615_200420_35_speaker_00.txt (3 segments)
Saved: le20250615_real_111424_113400055_20250615_200420_35_speaker_01.txt (1 segments)
Saved: le20250615_real_111424_113400055_20250615_200420_35_speaker_02.txt (2 segments)
Saved: le20250615_real_111424_113400055_20250615_200420_35_speaker_03.txt (8 segments)

DONE! 4 speakers identified.
All outputs in: /home/airsrg/work/video_streaming/output/le20250615_real_111424_113400055_20250615_200420_35_speakers/

  le20250615_real_111424_113400055_20250615_200420_35_diarized.json     2.3 KB
  le20250615_real_111424_113400055_20250615_200420_35_diarized.srt     1.2 KB
  le20250615_real_111424_113400055_20250615_200420_35_diarized.txt     1.0 KB
  le20250615_real_111424_113400055_20250615_200420